# 04 — Evaluation & Error Analysis

**Goal:** Load a saved model checkpoint, run predictions on held-out data,
compute quantitative metrics, and visually inspect where the model succeeds and fails.

---

## Why error analysis matters

A single accuracy or mIoU number tells you *how well* a model performs on average,
but not *why* it fails or *which* failure modes are most important to fix.

Common failure modes in Mars terrain segmentation:
- **Sand/soil confusion** — these classes look similar in colour and texture.
- **Missed small rocks** — low-resolution features are hard for shallow models.
- **Shadow regions** — dark areas may be misclassified as bedrock.
- **Boundary bleeding** — class boundaries are smeared, especially at coarse resolution.

> **Prerequisite:** Run `03_baseline_training.ipynb` first to save a checkpoint.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split

from src.data_paths import RAW_DATA_DIR, MODELS_DIR, ensure_project_dirs
from src.dataset import AI4MarsDataset, find_image_files, find_mask_files, build_pairs_by_stem
from src.train_utils import get_device, load_checkpoint
from src.metrics import pixel_accuracy, intersection_over_union, mean_iou
from src.visualize import CLASS_NAMES, show_image, show_mask, show_image_mask_overlay

ensure_project_dirs()

## Step 1 — Rebuild the Same Model Architecture

The checkpoint only stores **weights**, not the model class definition.  
You must define the same architecture here before loading.

In [ ]:
import segmentation_models_pytorch as smp

NUM_CLASSES = 4
IGNORE_INDEX = 255
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 4
VAL_SPLIT = 0.1

# Same weighted-loss setup used during training in notebook 03.
CLASS_WEIGHTS = torch.tensor([1.0, 2.5, 1.5, 4.0], dtype=torch.float32)

device = get_device()

def build_unet(encoder_name: str, num_classes: int, device: torch.device):
    return smp.Unet(
        encoder_name=encoder_name,
        encoder_weights="imagenet",
        in_channels=3,
        classes=num_classes,
    ).to(device)

def infer_encoder_name(state_dict: dict) -> str:
    # Checkpoint from 03 can be either resnet34 (CUDA path) or mobilenet_v2 (CPU path).
    keys = list(state_dict.keys())
    if any(k.startswith("encoder.layer1.") for k in keys):
        return "resnet34"
    if any(k.startswith("encoder.features.") for k in keys):
        return "mobilenet_v2"
    raise RuntimeError("Could not infer encoder type from checkpoint state_dict.")

## Step 2 — Load Checkpoint

In [ ]:
checkpoint_path = MODELS_DIR / "weighted_unet_epoch03.pth"

if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device)
    encoder_name = infer_encoder_name(checkpoint["model_state_dict"])

    model = build_unet(encoder_name, NUM_CLASSES, device)
    optimizer = torch.optim.Adam(model.parameters())  # needed by load_checkpoint

    epoch = load_checkpoint(model, optimizer, checkpoint_path, device)
    print(f"Loaded checkpoint from epoch {epoch}")
    print(f"Loaded model architecture: U-Net with encoder='{encoder_name}'")
else:
    raise FileNotFoundError(
        f"Checkpoint not found at {checkpoint_path}. "
        "Run 03_baseline_training.ipynb first."
    )

## Step 3 — Recreate the Validation Set

In [ ]:
DATA_ROOT = RAW_DATA_DIR

image_files = find_image_files(DATA_ROOT)
mask_files = find_mask_files(DATA_ROOT)
pairs = build_pairs_by_stem(image_files, mask_files)

# Match notebook 03 exactly: evaluate NAV-only pairs for the 4-class setup.
pairs = [(img, msk) for img, msk in pairs if "M2020_GEO" not in str(msk)]

total = len(pairs)
val_size = max(1, int(total * VAL_SPLIT))
train_size = total - val_size

full_dataset = AI4MarsDataset(pairs, image_size=IMAGE_SIZE)
_, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42),  # same seed as notebook 03
)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Validation samples (NAV-only): {len(val_dataset)}")

## Step 4 — Run Predictions and Compute Metrics

In [ ]:
# Safety first: free any cached CUDA memory from prior runs.
if torch.cuda.is_available():
    torch.cuda.empty_cache()

loss_fn = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(device), ignore_index=IGNORE_INDEX)

# 1) Smoke-test one batch to validate memory behavior before full eval.
model.eval()
with torch.no_grad():
    images, masks = next(iter(val_loader))
    images = images.to(device)
    outputs = model(images)

    # Move results back to CPU before further analysis/visualization.
    preds = outputs.argmax(dim=1).cpu()
    images = images.cpu()
    masks = masks.cpu()

print(f"Smoke test batch ok: images={tuple(images.shape)}, preds={tuple(preds.shape)}")

# 2) Optional full validation pass. Keep False while debugging OOM.
RUN_FULL_EVAL = False

if RUN_FULL_EVAL:
    all_preds = []
    all_targets = []
    total_loss = 0.0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)  # [B, num_classes, H, W]
            loss = loss_fn(logits, masks)
            total_loss += loss.item()

            # argmax converts logits -> predicted class ID per pixel
            pred_batch = logits.argmax(dim=1)  # [B, H, W]
            all_preds.append(pred_batch.cpu())
            all_targets.append(masks.cpu())

    all_preds = torch.cat(all_preds, dim=0)      # [N, H, W]
    all_targets = torch.cat(all_targets, dim=0)  # [N, H, W]

    mean_val_loss = total_loss / len(val_loader)
    pix_acc = pixel_accuracy(all_preds, all_targets, ignore_index=IGNORE_INDEX)
    per_class_iou = intersection_over_union(
        all_preds,
        all_targets,
        num_classes=NUM_CLASSES,
        ignore_index=IGNORE_INDEX,
    )
    m_iou = mean_iou(
        all_preds,
        all_targets,
        num_classes=NUM_CLASSES,
        ignore_index=IGNORE_INDEX,
    )

    print(f"Validation loss (weighted): {mean_val_loss:.4f}")
    print(f"Pixel accuracy            : {pix_acc:.4f}")
    print(f"Mean IoU                  : {m_iou:.4f}")
    print("\nPer-class IoU:")
    for c, iou in enumerate(per_class_iou):
        name = CLASS_NAMES.get(c, f"class_{c}")
        iou_str = f"{iou:.4f}" if iou is not None else "N/A (not in batch)"
        print(f"  class {c} ({name:>10s}): {iou_str}")
else:
    print("Skipped full evaluation (RUN_FULL_EVAL=False). Set True after smoke test passes.")

## Step 5 — Visual Prediction Comparison

For each sample: original image / ground-truth mask / predicted mask.

In [ ]:
max_samples = 8  # keep small to avoid heavy inline rendering

model.eval()
shown = 0

with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(device)
        outputs = model(images)

        # Move results back to CPU before visualization.
        preds = outputs.argmax(dim=1).cpu()
        images = images.cpu()
        masks = masks.cpu()

        for i in range(images.shape[0]):
            if shown >= max_samples:
                break

            # Convert tensors to numpy for visualisation
            img_np = images[i].permute(1, 2, 0).numpy()  # [H, W, 3] float32
            true_np = masks[i].numpy()                    # [H, W] int64
            pred_np = preds[i].numpy()                    # [H, W] int64

            fig, axes = plt.subplots(1, 3, figsize=(15, 5))
            show_image(img_np, title="Image", ax=axes[0])
            show_mask(true_np, title="Ground Truth", ax=axes[1])
            show_mask(pred_np, title="Prediction", ax=axes[2])
            plt.suptitle(f"Validation sample {shown + 1}", fontsize=11)
            plt.tight_layout()
            plt.show()
            plt.close(fig)
            shown += 1

        if shown >= max_samples:
            break

print(f"Displayed {shown} validation samples.")

## Step 6 — Failure Case Notes

Use this section to document observations after looking at the predictions above.

---

### Observations

| # | Failure type | Example samples | Likely cause |
|---|---|---|---|
| 1 | Sand/soil confusion | e.g. samples 2, 5 | Similar texture, lighting changes |
| 2 | Missed small rocks | e.g. sample 3 | Small-object recall still limited |
| 3 | Shadow misclassification | e.g. sample 1 | Illumination shift / low local contrast |
| 4 | ... | ... | ... |

### Ideas for Improvement

- [ ] Add stronger augmentation (flip, color jitter, contrast).
- [ ] Tune class weights using measured label frequencies from NAV masks.
- [ ] Train longer and use LR scheduling (e.g., cosine/one-cycle).
- [ ] Try larger input size for small rock recall if VRAM allows.
- [ ] Add overlay-only audits focused on the hardest IoU class.